# Fabric Data Agent MCP -> Copilot Studio Setup Notebook

This notebook walks through the complete setup needed to connect Fabric Data Agents (via their MCP endpoints) to a Copilot Studio agent that publishes to **M365 Copilot Chat**.

## What this notebook does
1. [OK] Verifies prerequisites (Azure CLI auth, Fabric access)
2. [SEARCH] Discovers your Fabric workspaces and Data Agents
3. [LINK] Retrieves and validates each Data Agent's MCP endpoint URL
4. [AUTH] Guides you through creating an Entra App Registration with the correct permissions
5. [TEST] Tests MCP endpoint connectivity with a live call
6. [NOTE] Generates the exact configuration block to paste into Copilot Studio

## Prerequisites
- Python 3.8+
- Azure CLI installed and logged in (`az login`)
- Your Fabric Data Agents must be **published** (Settings -> Model Context Protocol tab must be visible)
- Microsoft 365 Copilot license
- Fabric capacity F2 or higher

---
> **Architecture summary**: Fabric Data Agents natively expose MCP endpoints (Streamable HTTP + OAuth 2.0 via Entra ID). Copilot Studio connects to these via its MCP tool connector. This bypasses the broken Fabric connected-agent runtime that fails in M365 Copilot Chat.

## Step 0 - Install dependencies

In [ ]:
%pip install requests msal azure-identity tabulate --quiet
print("Dependencies installed.")

## Step 1 - Configuration

Fill in the values below. You can leave `WORKSPACE_IDS` empty to auto-discover **all** workspaces you have access to.

- `TENANT_ID`: Your Microsoft 365 / Entra tenant ID
- `WORKSPACE_IDS`: List of Fabric workspace IDs that contain your Data Agents (empty = scan all)
- `AGENT_NAMES_FILTER`: Optional list of Data Agent names to target (empty = include all agents found)

In [ ]:
# -- CONFIGURE THESE VALUES ----------------------------------------------------

TENANT_ID = ""                  # e.g. "72f988bf-86f1-41af-91ab-2d7cd011db47"
                                # Leave empty to auto-detect from az CLI

WORKSPACE_IDS = []              # e.g. ["<your-workspace-id>"]

AGENT_NAMES_FILTER = []         # e.g. ["Sales Agent", "HR Agent"]
                                # Leave empty to include all data agents found

# -----------------------------------------------------------------------------

# Fabric API base URL (stable)
FABRIC_API = "https://api.fabric.microsoft.com/v1"
FABRIC_SCOPE = "https://api.fabric.microsoft.com/.default"

print("Configuration loaded.")

## Step 2 - Authenticate with Azure CLI

This uses your existing `az login` session. If prompted, run `az login` in a terminal first.

In [ ]:
import subprocess
import json
import requests
from azure.identity import AzureCliCredential

credential = AzureCliCredential()

# Auto-detect tenant ID if not provided
if not TENANT_ID:
    result = subprocess.run(
        ["az", "account", "show", "--output", "json"],
        capture_output=True, text=True, check=True, shell=True
    )
    account = json.loads(result.stdout)
    TENANT_ID = account["tenantId"]
    print(f"Auto-detected Tenant ID : {TENANT_ID}")
    print(f"Subscription            : {account['name']} ({account['id']})")
    print(f"User                    : {account['user']['name']}")
else:
    print(f"Using Tenant ID: {TENANT_ID}")

# Get a Fabric API token
token = credential.get_token(FABRIC_SCOPE).token
HEADERS = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

print("\n[OK] Authenticated with Fabric API.")

## Step 3 - Discover Fabric Workspaces

In [ ]:
from tabulate import tabulate

def fabric_get(path, headers=None):
    """Helper: GET from Fabric API with auto token refresh."""
    h = headers or HEADERS
    # Refresh token if needed
    h["Authorization"] = f"Bearer {credential.get_token(FABRIC_SCOPE).token}"
    resp = requests.get(f"{FABRIC_API}{path}", headers=h)
    resp.raise_for_status()
    return resp.json()

if WORKSPACE_IDS:
    # Validate the provided workspace IDs
    workspaces = []
    for ws_id in WORKSPACE_IDS:
        try:
            ws = fabric_get(f"/workspaces/{ws_id}")
            workspaces.append({"id": ws["id"], "displayName": ws["displayName"], "capacityId": ws.get("capacityId", "N/A")})
        except Exception as e:
            print(f"[WARN]  Could not access workspace {ws_id}: {e}")
else:
    # Auto-discover all workspaces
    data = fabric_get("/workspaces")
    workspaces = [{"id": ws["id"], "displayName": ws["displayName"], "capacityId": ws.get("capacityId", "N/A")} for ws in data.get("value", [])]

print(f"Found {len(workspaces)} workspace(s):\n")
print(tabulate(workspaces, headers="keys", tablefmt="rounded_outline"))

## Step 4 - Discover Fabric Data Agents

Scans each workspace for published Data Agents.

In [ ]:
data_agents = []

for ws in workspaces:
    ws_id = ws["id"]
    ws_name = ws["displayName"]
    try:
        items = fabric_get(f"/workspaces/{ws_id}/items?type=DataAgent")
        agents = items.get("value", [])
        for agent in agents:
            if AGENT_NAMES_FILTER and agent["displayName"] not in AGENT_NAMES_FILTER:
                continue
            data_agents.append({
                "workspace_id": ws_id,
                "workspace_name": ws_name,
                "agent_id": agent["id"],
                "agent_name": agent["displayName"],
                "description": agent.get("description", "")[:60]
            })
    except requests.HTTPError as e:
        if e.response.status_code == 404:
            pass  # No data agents in this workspace
        else:
            print(f"[WARN]  Error scanning workspace '{ws_name}': {e}")

if not data_agents:
    print("[WARN]  No Data Agents found. Make sure your agents are published and you have access.")
else:
    print(f"Found {len(data_agents)} Data Agent(s):\n")
    display_rows = [{k: v for k, v in a.items() if k != "workspace_id"} for a in data_agents]
    print(tabulate(display_rows, headers="keys", tablefmt="rounded_outline"))

## Step 5 - Retrieve MCP Endpoint URLs

Each published Fabric Data Agent exposes an MCP endpoint. This cell builds and validates each URL.

> **Note**: The MCP endpoint URL is also visible in Fabric UI: Data Agent -> Settings -> **Model Context Protocol** tab.

In [ ]:
import re

def sanitize_server_name(name):
    """Copilot Studio MCP server names: alphanumeric, hyphens, underscores only."""
    sanitized = re.sub(r'[^a-zA-Z0-9_-]', '_', name)
    sanitized = re.sub(r'_+', '_', sanitized).strip('_')
    return sanitized

MCP_URL_TEMPLATE = "{fabric_api}/mcp/workspaces/{workspace_id}/dataagents/{agent_id}/agent"

mcp_endpoints = []

for agent in data_agents:
    mcp_url = MCP_URL_TEMPLATE.format(
        fabric_api=FABRIC_API,
        workspace_id=agent["workspace_id"],
        agent_id=agent["agent_id"]
    )
    
    # Validate the endpoint responds (MCP initialize handshake)
    status = "unknown"
    try:
        token_fresh = credential.get_token(FABRIC_SCOPE).token
        resp = requests.post(
            mcp_url,
            headers={
                "Authorization": f"Bearer {token_fresh}",
                "Content-Type": "application/json",
                "Accept": "application/json, text/event-stream"
            },
            json={
                "jsonrpc": "2.0",
                "id": 1,
                "method": "initialize",
                "params": {
                    "protocolVersion": "2025-03-26",
                    "capabilities": {},
                    "clientInfo": {"name": "setup-notebook", "version": "1.0"}
                }
            },
            timeout=15
        )
        if resp.status_code in (200, 202):
            status = "[OK] reachable"
        elif resp.status_code == 401:
            status = "[AUTH] auth required"
        elif resp.status_code == 404:
            status = "[ERROR] not found (is it published?)"
        else:
            status = f"[WARN]  HTTP {resp.status_code}"
    except requests.exceptions.Timeout:
        status = "[TIMEOUT] timeout"
    except Exception as e:
        status = f"[ERROR] error: {str(e)[:40]}"

    mcp_endpoints.append({
        "agent_name": agent["agent_name"],
        "server_name": sanitize_server_name(agent["agent_name"]),
        "workspace_name": agent["workspace_name"],
        "mcp_url": mcp_url,
        "status": status
    })

print("MCP Endpoint Validation Results:\n")
print(tabulate(mcp_endpoints, headers="keys", tablefmt="rounded_outline"))

## Step 6 - Create Entra App Registration

Copilot Studio needs an **Entra App Registration** to authenticate users against the Fabric MCP endpoints via OAuth 2.0. This cell creates one (or validates an existing one).

**Permissions required on the app registration:**
- `https://api.fabric.microsoft.com/DataAgent.Execute.All` (delegated)
- `https://api.fabric.microsoft.com/Item.Read.All` (delegated)

> [WARN] This step requires **Application Administrator** or **Cloud Application Administrator** role in Entra ID.

In [ ]:
import subprocess, json

APP_DISPLAY_NAME = "Copilot Studio - Fabric MCP Connector"

def az(cmd):
    result = subprocess.run(["az"] + cmd + ["--output", "json"], capture_output=True, text=True, shell=True)
    if result.returncode != 0:
        raise RuntimeError(f"az command failed: {result.stderr.strip()}")
    return json.loads(result.stdout) if result.stdout.strip() else {}

# Check if app already exists
existing = az(["ad", "app", "list", "--display-name", APP_DISPLAY_NAME, "--query", "[0]"] )

if existing:
    CLIENT_ID = existing["appId"]
    OBJECT_ID = existing["id"]
    print(f"[OK] App registration already exists.")
    print(f"   Display Name : {APP_DISPLAY_NAME}")
    print(f"   Client ID    : {CLIENT_ID}")
else:
    print(f"Creating app registration: '{APP_DISPLAY_NAME}'...")
    app = az([
        "ad", "app", "create",
        "--display-name", APP_DISPLAY_NAME,
        "--sign-in-audience", "AzureADMyOrg"
    ])
    CLIENT_ID = app["appId"]
    OBJECT_ID = app["id"]
    print(f"[OK] App registration created.")
    print(f"   Display Name : {APP_DISPLAY_NAME}")
    print(f"   Client ID    : {CLIENT_ID}")

    # Clear any existing (possibly wrong) permissions before adding correct ones
    print("   Clearing any existing API permissions on the app registration...")
    az(["ad", "app", "update", "--id", CLIENT_ID, "--required-resource-accesses", "[]"] )
    print("   Existing permissions cleared.")

    # Dynamically discover the Fabric SP by its API identifier URI - more reliable than display name
    print("   Looking up Fabric API service principal by identifier URI...")
    FABRIC_API_URI = "https://api.fabric.microsoft.com"
    sp_results = az(["ad", "sp", "list",
        "--filter", f"servicePrincipalNames/any(n:startswith(n,'{FABRIC_API_URI}'))",
        "--query", "[0]"
    ])
    if not sp_results:
        # Fallback: search by display name
        print("   URI search returned no results, falling back to display name...")
        for name in ["Microsoft Fabric", "Power BI Service"]:
            sp_results = az(["ad", "sp", "list", "--filter", f"displayName eq '{name}'", "--query", "[0]"] )
            if sp_results:
                break
    if not sp_results:
        raise RuntimeError("Could not find the Fabric API service principal. Check tenant access.")

    FABRIC_APP_ID = sp_results["appId"]
    print(f"   Found Fabric SP: '{sp_results['displayName']}' - App ID: {FABRIC_APP_ID}")

    # Look up exact scope IDs from the SP's app registration (not the SP object itself)
    # The app registration holds the canonical oauth2PermissionScopes with correct IDs
    REQUIRED_SCOPES = ["DataAgent.Execute.All", "Item.Read.All"]
    app_detail = az(["ad", "app", "show", "--id", FABRIC_APP_ID])
    available_scopes = app_detail.get("api", {}).get("oauth2PermissionScopes", [])
    if not available_scopes:
        # Some tenants surface scopes on the SP object instead
        sp_detail = az(["ad", "sp", "show", "--id", FABRIC_APP_ID])
        available_scopes = sp_detail.get("oauth2PermissionScopes", [])
    scope_map = {s["value"]: s["id"] for s in available_scopes if s.get("isEnabled", True)}
    print(f"   Available enabled delegated scopes ({len(scope_map)}): {list(scope_map.keys())}")

    missing = [s for s in REQUIRED_SCOPES if s not in scope_map]
    if missing:
        raise RuntimeError(f"Required scopes not found: {missing}.\nAvailable: {list(scope_map.keys())}")

    resource_access = [{"id": scope_map[s], "type": "Scope"} for s in REQUIRED_SCOPES]
    for s in REQUIRED_SCOPES:
        print(f"   [OK] Scope {s}: {scope_map[s]}")

    PERMISSIONS = [{"resourceAppId": FABRIC_APP_ID, "resourceAccess": resource_access}]

    import tempfile, os
    with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f:
        json.dump(PERMISSIONS, f)
        perm_file = f.name

    try:
        az(["ad", "app", "update", "--id", CLIENT_ID, "--required-resource-accesses", f"@{perm_file}"] )
        print("   Fabric API permissions added.")
    except Exception as e:
        print(f"   [WARN]  Could not add permissions automatically: {e}")
        print("   Add manually: Fabric API -> Delegated -> DataAgent.Execute.All, Item.Read.All")
    finally:
        os.unlink(perm_file)

print(f"\nClient ID for next steps: {CLIENT_ID}")

## Step 7 - Create a Client Secret

Copilot Studio's MCP connector (Manual OAuth 2.0) requires a client secret.

> The secret value is shown **only once**. It is stored in this notebook session only - save it to your secret store.

In [ ]:
from datetime import datetime, timezone

SECRET_DISPLAY_NAME = "copilot-studio-mcp"

# Check for existing secrets
existing_secrets = az(["ad", "app", "credential", "list", "--id", CLIENT_ID, "--query", "[?displayName=='copilot-studio-mcp']"])

if existing_secrets:
    print("[WARN]  A secret named 'copilot-studio-mcp' already exists.")
    print("   If you no longer have the value, delete it and re-run this cell.")
    for s in existing_secrets:
        exp = s.get('endDateTime', 'unknown')
        print(f"   Key ID: {s['keyId']}  |  Expires: {exp}")
    CLIENT_SECRET = input("\nPaste your existing secret value (or leave blank to create a new one): ").strip()
    
    if not CLIENT_SECRET:
        # Delete old and create new
        for s in existing_secrets:
            az(["ad", "app", "credential", "delete", "--id", CLIENT_ID, "--key-id", s["keyId"]])
        print("   Old secret deleted. Creating new one...")
        secret_result = az(["ad", "app", "credential", "reset", "--id", CLIENT_ID, "--display-name", SECRET_DISPLAY_NAME, "--years", "1", "--append"])
        CLIENT_SECRET = secret_result["password"]
        print(f"\n[OK] New client secret created.")
        print(f"   SECRET VALUE (save this now!): {CLIENT_SECRET}")
else:
    print(f"Creating client secret '{SECRET_DISPLAY_NAME}'...")
    secret_result = az(["ad", "app", "credential", "reset", "--id", CLIENT_ID, "--display-name", SECRET_DISPLAY_NAME, "--years", "1", "--append"])
    CLIENT_SECRET = secret_result["password"]
    print(f"[OK] Client secret created.")
    print(f"   SECRET VALUE (save this now!): {CLIENT_SECRET}")

## Step 8 - Grant Admin Consent

The delegated Fabric permissions need **admin consent** so users don't see a consent prompt when they first use the agent.

In [ ]:
# Create service principal (required for admin consent) - or retrieve if it already exists
try:
    sp = az(["ad", "sp", "create", "--id", CLIENT_ID])
    SP_OBJECT_ID = sp["id"]
    print(f"[OK] Service principal created: {SP_OBJECT_ID}")
except RuntimeError:
    # SP already exists - retrieve it directly by appId (most reliable method)
    sp = az(["ad", "sp", "show", "--id", CLIENT_ID])
    SP_OBJECT_ID = sp["id"]
    print(f"[OK] Service principal already exists: {SP_OBJECT_ID}")

# Grant admin consent
try:
    az(["ad", "app", "permission", "admin-consent", "--id", CLIENT_ID])
    print("[OK] Admin consent granted for Fabric API delegated permissions.")
except Exception as e:
    print(f"[WARN]  Admin consent via CLI failed: {e}")
    print("\n[NOTE] Grant consent manually:")
    print(f"   1. Go to: https://entra.microsoft.com/#blade/Microsoft_AAD_RegisteredApps/ApplicationMenuBlade/CallAnAPI/appId/{CLIENT_ID}")
    print(f"   2. Click 'Grant admin consent for <your org>'")

## Step 9 - Test OAuth Token Exchange

Verifies the app registration can obtain a Fabric API token using the client credentials flow.

In [ ]:
import msal
import subprocess, json

# Re-detect TENANT_ID if not set (guards against kernel restarts)
if not TENANT_ID:
    _acct = json.loads(subprocess.run(["az", "account", "show", "--output", "json"], capture_output=True, text=True, shell=True).stdout)
    TENANT_ID = _acct["tenantId"]
    print(f"Re-detected Tenant ID: {TENANT_ID}")

AUTHORITY = f"https://login.microsoftonline.com/{TENANT_ID}"
FABRIC_SCOPE_USER = "https://api.fabric.microsoft.com/DataAgent.Execute.All"

# Use client credentials (app-only) to validate the registration is correct
# Note: in production Copilot Studio uses delegated (user) auth
msal_app = msal.ConfidentialClientApplication(
    client_id=CLIENT_ID,
    client_credential=CLIENT_SECRET,
    authority=AUTHORITY
)

result = msal_app.acquire_token_for_client(scopes=[FABRIC_SCOPE])

if "access_token" in result:
    print("[OK] Token acquired successfully.")
    print(f"   Token type   : {result.get('token_type')}")
    print(f"   Expires in   : {result.get('expires_in')}s ({result.get('expires_in', 0)//60} minutes)")
    print(f"   Scope        : {result.get('scope', 'N/A')}")
else:
    print(f"[ERROR] Token acquisition failed.")
    print(f"   Error        : {result.get('error')}")
    print(f"   Description  : {result.get('error_description')}")
    print("\n   Troubleshooting:")
    print("   - Verify admin consent was granted")
    print("   - Verify the client secret is correct")
    print("   - Verify the Fabric API permissions are configured")

## Step 10 - Generate Copilot Studio Configuration

This cell outputs the **exact values** you need to enter in Copilot Studio when adding each Fabric Data Agent as an MCP tool.

**Where to enter these in Copilot Studio:**
> Agent -> Tools -> Add a tool -> New tool -> Model Context Protocol -> fill in fields below

In [ ]:
import re

def sanitize_server_name(name):
    """MCP server names in Copilot Studio only allow letters, digits, - and _"""
    return re.sub(r'[^A-Za-z0-9_-]', '_', name)

AUTH_URL     = f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/authorize"
TOKEN_URL    = f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token"
REFRESH_URL  = TOKEN_URL
SCOPES       = "https://api.fabric.microsoft.com/DataAgent.Execute.All https://api.fabric.microsoft.com/Item.Read.All openid profile"

separator = "-" * 70

print("\n" + "=" * 70)
print(" COPILOT STUDIO - MCP TOOL CONFIGURATION")
print("=" * 70)
print()
print("SHARED OAUTH 2.0 SETTINGS (same for all 3 agents)")
print(separator)
print(f"  Authentication type : OAuth 2.0")
print(f"  OAuth 2.0 type      : Manual")
print(f"  Client ID           : {CLIENT_ID}")
print(f"  Client Secret       : {'*' * 8} (value saved from Step 7)")
print(f"  Authorization URL   : {AUTH_URL}")
print(f"  Token URL           : {TOKEN_URL}")
print(f"  Refresh URL         : {REFRESH_URL}")
print(f"  Scopes              : {SCOPES}")
print()

for i, ep in enumerate(mcp_endpoints, 1):
    print(f"AGENT {i}: {ep['agent_name']}  [{ep['workspace_name']}]")
    print(separator)
    print(f"  Server name         : {ep['server_name']}")
    print(f"  Server description  : Fabric Data Agent for {ep['agent_name']}. ")
    print(f"                        Use this agent to answer analytics questions")
    print(f"                        about {ep['agent_name'].lower()} data.")
    print(f"  Server URL          : {ep['mcp_url']}")
    print(f"  Endpoint test       : {ep['status']}")
    print()

print("=" * 70)
print(" AFTER ADDING EACH MCP TOOL IN COPILOT STUDIO:")
print(separator)
print(" 1. Copilot Studio shows a Redirect URI - copy it")
print(" 2. Run the cell below to add it to the app registration")
print("=" * 70)

## Step 11 - Add Copilot Studio Redirect URIs to App Registration

After you create the MCP tool connection in Copilot Studio, it displays a **Redirect URI** (callback URL). Add it here to complete the OAuth flow.

In [ ]:
# Paste the redirect URI(s) from Copilot Studio here - each must be a quoted string
# Example:
# REDIRECT_URIS = [
#     "https://global.consent.azure-apim.net/redirect/your-connector-id",
# ]
# You get one per MCP tool connection created in Copilot Studio
REDIRECT_URIS = [
    # "https://global.consent.azure-apim.net/redirect/your-connector-id",
    # Add more as needed, one per agent connection
]

# Normalize URIs (drop blanks, remove duplicates while preserving order)
cleaned_redirect_uris = []
for uri in REDIRECT_URIS:
    candidate = uri.strip()
    if candidate and candidate not in cleaned_redirect_uris:
        cleaned_redirect_uris.append(candidate)

# Re-detect CLIENT_ID if kernel was restarted
if not CLIENT_ID:
    _apps = az(["ad", "app", "list", "--display-name", "Copilot Studio - Fabric MCP Connector"] )
    CLIENT_ID = _apps[0]["appId"] if _apps else ""
    print(f"Re-detected CLIENT_ID: {CLIENT_ID}")

if not CLIENT_ID:
    raise RuntimeError("CLIENT_ID is empty. Run Step 6 first to create or locate the app registration.")

if not cleaned_redirect_uris:
    print("[INFO]  No redirect URIs provided yet.")
    print("   Complete Step 10 in Copilot Studio first, then paste the Redirect URI(s) above.")
else:
    # Get current redirect URIs
    app_info = az(["ad", "app", "show", "--id", CLIENT_ID])
    existing_uris = app_info.get("web", {}).get("redirectUris", [])

    all_uris = []
    for uri in existing_uris + cleaned_redirect_uris:
        if uri not in all_uris:
            all_uris.append(uri)

    az(["ad", "app", "update", "--id", CLIENT_ID, "--web-redirect-uris"] + all_uris)

    print("[OK] Redirect URIs updated on app registration.")
    for uri in all_uris:
        marker = "(new)" if uri in cleaned_redirect_uris else "(existing)"
        print(f"   {marker} {uri}")

## Step 12 - Validate End-to-End MCP Tool Call

Makes a test call to each MCP endpoint simulating what Copilot Studio will do at runtime.

In [ ]:
TEST_QUERY = "What data is available?"  # Simple probe question

print("Testing MCP tool/list (capability discovery) for each agent...\n")

for ep in mcp_endpoints:
    print(f"Agent: {ep['agent_name']}")
    print(f"URL  : {ep['mcp_url']}")
    
    try:
        fresh_token = credential.get_token(FABRIC_SCOPE).token
        
        # MCP tools/list request
        resp = requests.post(
            ep["mcp_url"],
            headers={
                "Authorization": f"Bearer {fresh_token}",
                "Content-Type": "application/json",
                "Accept": "application/json, text/event-stream"
            },
            json={
                "jsonrpc": "2.0",
                "id": 2,
                "method": "tools/list",
                "params": {}
            },
            timeout=20
        )
        
        if resp.status_code == 200:
            body = resp.json()
            tools = body.get("result", {}).get("tools", [])
            print(f"[OK] Response: {resp.status_code} | Tools exposed: {len(tools)}")
            for t in tools:
                print(f"   Tool: {t.get('name')} - {t.get('description', '')[:80]}")
        else:
            # Handle SSE streaming response
            print(f"   HTTP {resp.status_code} | Content-Type: {resp.headers.get('Content-Type')}")
            print(f"   Response preview: {resp.text[:200]}")
    except Exception as e:
        print(f"[ERROR] Error: {e}")
    
    print()

## Step 13 - Summary & Next Steps

Generates a final summary and checklist for completing the Copilot Studio configuration.

In [ ]:
print("=" * 70)
print(" SETUP SUMMARY")
print("=" * 70)
print(f"  Tenant ID          : {TENANT_ID}")
print(f"  App Registration   : {APP_DISPLAY_NAME}")
print(f"  Client ID          : {CLIENT_ID}")
print(f"  Data Agents Found  : {len(mcp_endpoints)}")
print()
print("MCP Endpoints:")
for ep in mcp_endpoints:
    print(f"  [{ep['status']}] {ep['agent_name']}")
    print(f"    {ep['mcp_url']}")

print()
print("-" * 70)
print(" REMAINING STEPS IN COPILOT STUDIO")
print("-" * 70)
print()
print(" IN COPILOT STUDIO (copilotstudio.microsoft.com):")
print()
print(" [ ] 1. Create a new agent (or open existing)")
print(" [ ] 2. For EACH of your 3 Fabric Data Agents:")
print("        Tools -> Add a tool -> New tool -> Model Context Protocol")
print("        Paste Server Name, Description, URL from Step 10 output")
print("        Auth: OAuth 2.0 -> Manual -> paste values from Step 10")
print("        Copy the Redirect URI shown -> paste into Step 11 above")
print("        Select: Create -> Next -> Create new connection -> Sign in")
print()
print(" [ ] 3. Add SharePoint knowledge source:")
print("        Knowledge -> Add knowledge -> SharePoint")
print("        Paste your SharePoint Document Library URL")
print()
print(" [ ] 4. Add agent instructions (system prompt) that:")
print("        - Routes analytics questions -> Fabric MCP tools")
print("        - Routes document/qualitative questions -> SharePoint")
print("        - Describes chart generation behavior")
print()
print(" [ ] 5. Configure chart generation:")
print("        Option A: Use Adaptive Cards (JSON in agent response)")
print("        Option B: Return Power BI embed URL from MCP tool call")
print("        Option C: Add an Azure Function MCP tool for Chart.js SVG")
print()
print(" [ ] 6. Publish:")
print("        Channels -> Teams and Microsoft 365 -> Add channel")
print("        This surfaces the agent in M365 Copilot Chat UI")
print()
print("=" * 70)

## Appendix A - Agent System Prompt Template

Copy this as a starting point for your agent's instructions in Copilot Studio.

In [ ]:
agent_names = [ep['agent_name'] for ep in mcp_endpoints]
agent_list  = "\n".join([f"  - {n}" for n in agent_names])

SYSTEM_PROMPT = f"""You are an enterprise analytics assistant. You help users explore data and documents.

## Data Sources

You have access to the following Fabric Data Agents via MCP:
{agent_list}

You also have access to a SharePoint Document Library for qualitative and policy documents.

## Routing Logic

- For questions about metrics, numbers, trends, KPIs, or analytics -> use the appropriate Fabric Data Agent MCP tool.
- For questions about documents, policies, procedures, or qualitative topics -> use SharePoint knowledge.
- If a question spans both, retrieve data from both sources and synthesize a single response.

## Selecting the right Fabric Agent

Examine the user's question and choose the most relevant Fabric Data Agent based on its name and description.
If multiple agents might apply, invoke the most specific one first.

## Charts and Visualizations

When a user asks for a chart, graph, or visualization:
1. Retrieve the underlying data from the appropriate Fabric Data Agent.
2. Present the data in a clear table format.
3. Describe the trend or pattern in natural language.
4. If the user requests a Power BI report, provide the direct link to the relevant report.

## Response Style

- Be concise and direct.
- Always cite which data source you used.
- If data is unavailable or the query fails, explain clearly and suggest alternatives.
"""

print("AGENT SYSTEM PROMPT (paste into Copilot Studio -> Instructions):")
print("-" * 70)
print(SYSTEM_PROMPT)

## Appendix B - Troubleshooting Reference

| Symptom | Likely Cause | Fix |
|---|---|---|
| MCP endpoint returns 404 | Data Agent not published | Publish it in Fabric UI: Data Agent -> Publish |
| MCP endpoint returns 401 | Token scope wrong or admin consent missing | Re-run Step 8, verify scopes |
| Copilot Studio shows 'connection failed' | Redirect URI not registered | Run Step 11 with the URI from Copilot Studio |
| Agent works in Teams but not M365 Chat | Using native Fabric connector (not MCP) | Ensure you used Tools -> MCP, not Agents -> Fabric |
| Agent picks wrong Fabric agent | Vague tool descriptions | Edit the Server Description in Copilot Studio to be specific |
| No charts rendered | Agent lacks chart capability | Use Appendix A system prompt or add Azure Function MCP tool |